# Spektralgruppen einzeln betrachten (#34)

**Anlass:** Die Permutation Importance in `09_analyse/02_duac1011_feature-importance.ipynb` mischt jede Gruppe im Modell, das **alle** 140 Merkmale kennt. Das Feedback: Gruppen auch **einzeln** betrachten, weil Redundanz und unterschiedliche Gruppengrößen das Bild verzerren, etwa bei der Red Edge (7 Bänder).

**Vorgehen:** Nur der Trainingsanteil und seine gemeinsamen CV-Folds; die Validierung bleibt unberührt. Jede Variante bekommt eine eigene kleine Suche (`tune()`, MLflow-Experiment `spectral-groups`), weil ein Modell mit wenigen Bändern eine andere Kernelbreite braucht als eines mit 131.


In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
from functools import partial

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from joblib import Memory

from awp2.config import (
    ALL_BANDS_VARIANT,
    BAND_GROUPING,
    BAND_PREFIX,
    COMBINED_METRIC,
    GROUP_STUDY_APPROACH,
    GROUP_STUDY_CACHE_DIR,
    META_ONLY_VARIANT,
    METRIC_KIND,
    MODEL_DOCS_FIGURES_DIR,
    NULL_QUANTILE,
    ONLY_GROUP_KIND,
    ONLY_GROUP_NO_META_KIND,
    PERMUTATION_REPEATS,
    RANDOM_BAND_GROUP_COUNT,
    REGION_GROUPING,
    REGION_PERMUTATION_REPEATS,
    SPECTRAL_REGIONS,
    WITHOUT_GROUP_KIND,
)
from awp2.data import load_split_config
from awp2.experiment import RunConfig
from awp2.explain import (
    feature_groups,
    permutation_importance,
    random_band_groups,
    size_matched_windows,
)
from awp2.group_study import (
    VariantResult,
    group_variants,
    paired_difference,
    results_table,
    run_group_study,
    svm_param_grid,
    variant_preprocessing,
)
from awp2.models import make_combined_svm
from awp2.plots import (
    plot_group_against_null,
    plot_group_study,
    plot_importance_spectrum,
    save_doc_figure,
)
from awp2.preprocessing import PreprocessingConfig

# scale=True is the only flag that also standardises Month, which the reference model needs;
# it is deprecated and warns on every preprocessor build.
warnings.filterwarnings("ignore", category=DeprecationWarning)

REFERENCE_PREPROCESSING = PreprocessingConfig(scale=True, use_meta=True)
COMBINED_KEY = (METRIC_KIND, COMBINED_METRIC)
region_names = [region.name for region in SPECTRAL_REGIONS]

# Each variant and permutation is stored once, so an interrupted run continues where it stopped.
memory = Memory(GROUP_STUDY_CACHE_DIR, verbose=0)
cached_study = memory.cache(run_group_study)
cached_permutation = memory.cache(permutation_importance)
print(f"Split: Validierungsanteil {load_split_config().validation_size}")

## 1. Gruppen neu trainieren

Pro Spektralregion drei Modelle: **nur diese Region** (ohne und mit Metadaten) und **alle anderen Regionen** (mit Metadaten). Dazu als Referenzen alle Bänder (mit und ohne Metadaten) und nur die Metadaten. Gesucht wird `C` ∈ {10, 100, 1000} und `gamma` ∈ {0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1}; entschieden wird nur nach der CV-BAcc kombiniert. Die angegebenen CV-Werte sind das Maximum über 21 Kandidaten und damit absolut leicht optimistisch; alle Varianten werden gleich behandelt.


In [ ]:
results = [
    cached_study(
        make_combined_svm(), REFERENCE_PREPROCESSING, svm_param_grid(), variants=[variant]
    )[0]
    for variant in group_variants()
]
by_name = {result.variant.name: result for result in results}
table = results_table(results)
table.drop(columns=["tuning_run", "group"]).round(3)

In [ ]:
retraining_fig, ax = plt.subplots(figsize=(10, 5))
plot_group_study(table, ax=ax)
ax.set_title(
    "Referenz-SVM auf einzelnen Spektralregionen neu trainiert (CV auf dem Trainingsanteil)"
)
retraining_fig.tight_layout()
save_doc_figure(retraining_fig, "spectral_groups_retraining", MODEL_DOCS_FIGURES_DIR)

**Gepaarte Differenzen je Fold.** Alle Varianten laufen auf denselben fünf Folds. Die Differenz je Fold entfernt die Streuung, die beide teilen, und ist ein besserer Maßstab als die Standardabweichung der absoluten Werte. Bei fünf Folds bleibt sie ein Hinweis, kein Test.


In [ ]:
def region_result(kind: str, region: str) -> VariantResult:
    return next(r for r in results if r.variant.kind == kind and r.variant.group == region)


def paired_row(first: VariantResult, second: VariantResult) -> dict[str, float | str]:
    difference = paired_difference(first, second)
    return {
        "Differenz": difference.mean,
        "Std": difference.std,
        "Folds besser": f"{difference.folds_first_better}/{difference.n_folds}",
    }


reference = by_name[ALL_BANDS_VARIANT]
without_region = pd.DataFrame(
    {
        region: paired_row(region_result(WITHOUT_GROUP_KIND, region), reference)
        for region in region_names
    }
).T
print("Ohne Region minus Referenz (alle Bänder + Metadaten):")
display(without_region.round(3))

red_edge_with_meta = region_result(ONLY_GROUP_KIND, "Red Edge")
red_edge_alone = region_result(ONLY_GROUP_NO_META_KIND, "Red Edge")
others = [region for region in region_names if region != "Red Edge"]
red_edge_vs_others = pd.concat(
    {
        "mit Metadaten": pd.DataFrame(
            {r: paired_row(red_edge_with_meta, region_result(ONLY_GROUP_KIND, r)) for r in others}
        ).T,
        "ohne Metadaten": pd.DataFrame(
            {
                r: paired_row(red_edge_alone, region_result(ONLY_GROUP_NO_META_KIND, r))
                for r in others
            }
        ).T,
    },
    axis=1,
)
print("Nur Red Edge minus nur andere Region:")
display(red_edge_vs_others.round(3))

meta_only = by_name[META_ONLY_VARIANT]
gain_over_meta = pd.DataFrame(
    {
        region: paired_row(region_result(ONLY_GROUP_KIND, region), meta_only)
        for region in region_names
    }
).T
print("Nur Region + Metadaten minus nur Metadaten:")
display(gain_over_meta.round(3))

### Ergebnis: neu trainiert

Referenz (alle Bänder + Metadaten): CV-BAcc kombiniert **0.860 ± 0.027**; ohne Metadaten 0.786, nur Metadaten 0.536.

- **Keine Region ist allein unverzichtbar:** Ohne eine Region liegt die BAcc zwischen −0.007 und +0.003 von der Referenz (Std der Fold-Differenzen 0.006–0.018). Bei NIR und SWIR 1 ist das Modell ohne die Region in 4 von 5 Folds schlechter, aber nur um 0.005–0.007.
- **Mit Metadaten:** Die Red Edge erreicht mit nur **7 Bändern 0.792**. Gegenüber SWIR 1 (27 Bänder, 0.787) liegt sie gleichauf (+0.005, 2 von 5 Folds besser). Das NIR (44 Bänder, 0.811) ist in allen 5 Folds besser, aber nur um 0.018. Vor Blau, Rot, Grün und SWIR 2 liegt die Red Edge in **allen 5 Folds** (+0.027 bis +0.078).
- **Ohne Metadaten:** NIR 0.666 ist klar vorn (Red Edge 0.522, −0.144, 0 von 5 Folds). Die Red Edge liegt vor Blau, Rot, Grün und SWIR 2 in allen 5 Folds; gegenüber SWIR 1 (0.550) ist sie leicht unterlegen (−0.027, 1 von 5).
- **Gewinn über die Metadaten allein** (0.536): bei jeder Region positiv in allen 5 Folds, 0.178 (SWIR 2) bis 0.274 (NIR).
- **Folge für das Feedback:** Mit nur 7 Bändern trägt die Red Edge fast so viel wie die 6-mal größere NIR-Region und fällt im Gesamtmodell trotzdem nicht auf, weil NIR und SWIR sie ersetzen können.


## 2. Permutation im Gesamtmodell gegen Zufallsfenster gleicher Größe

Jede Region wird mit zufälligen Fenstern aus **gleich vielen benachbarten Bändern** verglichen, die die Region **nicht berühren** (sonst wäre der Vergleich zirkulär). Ist die Region wichtiger als das typische Fenster, liegt das nicht nur an ihrer Größe. Wo die Region fast alle Bänder umfasst (NIR: 44 von 131), gibt es nur wenige Fenster; die Anzahl steht in der Tabelle.


In [ ]:
reference_model = make_combined_svm().set_params(**reference.best_params)
reference_config = RunConfig(
    name="spectral_groups_importance",
    approach=GROUP_STUDY_APPROACH,
    preprocessing=REFERENCE_PREPROCESSING,
    track=False,
)

regions = cached_permutation(
    reference_model, reference_config, REGION_GROUPING, n_repeats=REGION_PERMUTATION_REPEATS
)
observed = regions.mean.loc[region_names, COMBINED_KEY]
band_counts = table[table["kind"] == ONLY_GROUP_NO_META_KIND].set_index("group")["n_bands"]

null = {}
for name in region_names:
    windows = cached_permutation(
        reference_model,
        reference_config,
        n_repeats=PERMUTATION_REPEATS,
        build_groups=partial(size_matched_windows, region=name, n_groups=RANDOM_BAND_GROUP_COUNT),
    )
    null[name] = windows.mean[COMBINED_KEY].reset_index(drop=True)
null = pd.DataFrame(null)


def scattered_outside_red_edge(columns: list[str]) -> dict[str, list[str]]:
    members = feature_groups(columns, REGION_GROUPING)["Red Edge"]
    return random_band_groups(
        columns, len(members), RANDOM_BAND_GROUP_COUNT, contiguous=False, exclude=members
    )


scattered = cached_permutation(
    reference_model,
    reference_config,
    n_repeats=PERMUTATION_REPEATS,
    build_groups=scattered_outside_red_edge,
).mean[COMBINED_KEY]

null_summary = pd.DataFrame(
    {
        "Bänder": band_counts[region_names],
        "Fenster": null.count(),
        "Region": observed,
        "Fenster Median": null.median(),
        f"Fenster {NULL_QUANTILE:.0%}": null.quantile(NULL_QUANTILE),
        "Fenster Min": null.min(),
        "Fenster Max": null.max(),
    }
)
null_summary["Region / Median"] = null_summary["Region"] / null_summary["Fenster Median"]
null_summary["Anteil Fenster unter Region"] = pd.Series(
    {name: (null[name].dropna() < observed[name]).mean() for name in region_names}
)
display(null_summary.round(3))
print(
    f"Verstreute Zufallsgruppen mit {int(band_counts['Red Edge'])} Bändern außerhalb der Red Edge: "
    f"Median {scattered.median():.3f}, Maximum {scattered.max():.3f}"
)

In [ ]:
null_fig, ax = plt.subplots(figsize=(10, 4.5))
plot_group_against_null(observed, null, ax=ax)
ax.set_title("Region gegen Zufallsfenster gleicher Bandzahl außerhalb der Region (Gesamtmodell)")
null_fig.tight_layout()
save_doc_figure(null_fig, "spectral_groups_null", MODEL_DOCS_FIGURES_DIR)

### Ergebnis: Region gegen Zufallsfenster

- **Die Red Edge ist pro Band wichtiger als ein typisches Fenster:** Abfall 0.209 gegenüber einem Median von 0.147 für Fenster aus 7 benachbarten Bändern außerhalb der Region; 39 von 40 Fenstern liegen darunter (95-%-Wert 0.192). Blau (0.225, 97.5 %) und Rot (0.220, 92.5 %) verhalten sich ähnlich, Grün (0.191, 80 %) nicht.
- **Einschränkung:** Die Vergleichsfenster liegen überwiegend im NIR und SWIR, wo viele Nachbarbänder einander ersetzen. Gezeigt ist, dass Bänder im Sichtbaren und in der Red Edge weniger ersetzbar sind, nicht, dass die Red Edge unter den schmalen Regionen einzigartig ist.
- **NIR:** Nur 13 Fenster gleicher Größe liegen außerhalb des NIR (alle im SWIR); sie sind alle weniger wichtig (Median 0.368 gegenüber 0.478). Das zeigt NIR > SWIR, aber keinen Größeneffekt.
- **SWIR 1 und SWIR 2** (0.340 und 0.343) liegen unter dem Median gleich großer Fenster außerhalb (0.438 und 0.396), die NIR-Bänder enthalten. Ihr hoher Wert in der Regionen-Permutation ist im Wesentlichen ein Größeneffekt.
- Verstreute Zufallsgruppen aus 7 Bändern außerhalb der Red Edge kosten nur 0.073 (Maximum 0.145): Zusammenhängende Fenster zerstören mehr, weil die Nachbarn fehlen.
- Die Streuung der Fenster ist groß (bei 7 Bändern 0.09–0.23) und die Fenster überlappen einander; das Verfahren trennt nur grob.


## 3. Permutation innerhalb einer Region

Das Modell sieht **nur eine Region** (ohne Metadaten) und jedes Band darin wird einzeln gemischt. Nachbarregionen können die Information nicht mehr ersetzen. Verglichen wird mit der Einzelband-Permutation im Gesamtmodell.


In [ ]:
within = {}
for result in results:
    if result.variant.kind != ONLY_GROUP_NO_META_KIND:
        continue
    config = RunConfig(
        name="spectral_groups_within",
        approach=GROUP_STUDY_APPROACH,
        preprocessing=variant_preprocessing(REFERENCE_PREPROCESSING, result.variant),
        track=False,
    )
    within[result.variant.group] = cached_permutation(
        make_combined_svm().set_params(**result.best_params),
        config,
        BAND_GROUPING,
        n_repeats=PERMUTATION_REPEATS,
    )

within_importance = pd.concat(
    [
        r.mean[COMBINED_KEY].rename("drop").to_frame().assign(region=name)
        for name, r in within.items()
    ]
)
within_std = pd.concat([r.std[COMBINED_KEY] for r in within.values()])

full_bands = cached_permutation(
    reference_model, reference_config, BAND_GROUPING, n_repeats=PERMUTATION_REPEATS
)
band_names = [name for name in full_bands.mean.index if name.startswith(BAND_PREFIX)]
full_importance = full_bands.mean.loc[band_names, COMBINED_KEY]
full_std = full_bands.std.loc[band_names, COMBINED_KEY]

within_by_region = within_importance.groupby("region")["drop"]
comparison = pd.DataFrame(
    {
        "Ø Band im Gesamtmodell": full_importance.groupby(within_importance["region"]).mean(),
        "Ø Band im Regionsmodell": within_by_region.mean(),
        "Min Band im Regionsmodell": within_by_region.min(),
        "Max Band im Regionsmodell": within_by_region.max(),
        "Baseline Regionsmodell": pd.Series(
            {name: r.baseline[COMBINED_KEY] for name, r in within.items()}
        ),
    }
).loc[region_names]
comparison["Faktor"] = comparison["Ø Band im Regionsmodell"] / comparison["Ø Band im Gesamtmodell"]
display(comparison.round(3))

print("Red-Edge-Modell, Abfall je Band (± Std über Folds und Wiederholungen):")
red_edge_bands = within_importance[within_importance["region"] == "Red Edge"]["drop"]
display(
    pd.DataFrame({"Abfall": red_edge_bands, "Std": within_std.reindex(red_edge_bands.index)}).round(
        3
    )
)
print("Je Region die drei Bänder mit dem größten Abfall im Regionsmodell:")
display(within_by_region.nlargest(3).round(3))

In [ ]:
within_fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
plot_importance_spectrum(
    full_importance, full_std, ax=axes[0], title="Gesamtmodell: jedes Band einzeln gemischt"
)
plot_importance_spectrum(
    within_importance["drop"],
    within_std.reindex(within_importance.index),
    ax=axes[1],
    title="Je Region ein eigenes Modell: jedes Band einzeln gemischt",
)
within_fig.tight_layout()
save_doc_figure(within_fig, "spectral_groups_within", MODEL_DOCS_FIGURES_DIR)

### Ergebnis: innerhalb einer Region

- Im Gesamtmodell kostet das Mischen eines Bandes im Mittel 0.007–0.026, im Modell **nur mit der Region** 0.19–0.37: je nach Region das 8,5- (Blau) bis 44-Fache (Rot). Die Redundanz zwischen den Regionen ist der Hauptgrund für die kleinen Einzelwerte.
- Im Red-Edge-Modell (Baseline 0.52) kostet jedes der sieben Bänder 0.33–0.42, also 64–81 % der Leistung; am meisten 681 nm (0.42), dann 722 nm (0.38) und 712 nm (0.37). Das Modell nutzt das **Zusammenspiel** der Bänder.
- Die Werte sind **nicht zwischen Regionen vergleichbar**: Je weniger Bänder ein Modell hat, desto weniger kann ein gemischtes Band ersetzt werden.
- Die wichtigsten Bänder liegen oft am Rand der Region oder neben einer Lücke (z. B. 681 nm, 752 nm, 1306/1316 nm, 1760 nm). Ob das ein Randeffekt oder echte Übergangsinformation ist, bleibt offen.


## 4. Gesamtübersicht je Region


In [ ]:
def score_by_region(kind: str, metric: str = COMBINED_METRIC) -> pd.Series:
    return table[table["kind"] == kind].set_index("group")[metric]


full_score = table.loc[ALL_BANDS_VARIANT, COMBINED_METRIC]
overview = pd.DataFrame(
    {
        "Bänder": band_counts,
        "Permutation Gesamtmodell": observed,
        "nur Region (ohne Meta)": score_by_region(ONLY_GROUP_NO_META_KIND),
        "nur Region + Meta": score_by_region(ONLY_GROUP_KIND),
        "ohne Region": score_by_region(WITHOUT_GROUP_KIND),
    }
).loc[region_names]
overview["Verlust ohne Region"] = full_score - overview["ohne Region"]
overview["Gewinn über Meta allein"] = (
    overview["nur Region + Meta"] - table.loc[META_ONLY_VARIANT, COMBINED_METRIC]
)
overview["Gewinn je Band"] = overview["Gewinn über Meta allein"] / overview["Bänder"]
display(overview.round(3))

print("Kultur und Stadium der Varianten mit Metadaten (BAcc):")
display(
    pd.DataFrame(
        {
            "Kultur": score_by_region(ONLY_GROUP_KIND, "bacc_crop"),
            "Stadium": score_by_region(ONLY_GROUP_KIND, "bacc_stage"),
        }
    )
    .loc[region_names]
    .round(3)
)

### Fazit

1. **Das Feedback trifft zu.** Im Gesamtmodell fällt die Red Edge absolut kaum auf (Abfall 0.209 gegenüber 0.478 im NIR). Pro Band ist sie aber wichtiger als 39 von 40 gleich großen Fenstern außerhalb, und mit nur 7 Bändern erreicht sie mit Metadaten 0.792 (NIR mit 44 Bändern 0.811, SWIR 1 mit 27 Bändern 0.787).
2. **Die Ursache ist Redundanz und Größe, nicht fehlende Information.** Ohne eine einzelne Region verliert das Modell höchstens 0.007.
3. **Die Red Edge ist nicht einzigartig:** Blau und Rot zeigen im Zufallsvergleich dasselbe Muster, und alle Regionen bringen über die Metadaten hinaus 0.18–0.27. Ohne Metadaten ist das NIR klar vorn.
4. **Konsequenz für die Bandauswahl (#126):** Eine Region nicht streichen, nur weil ihre Permutation klein ist – Gruppen immer zusätzlich allein, ohne sie und gegen Zufallsfenster prüfen.
5. **Grenzen:** 5 Folds, die Differenzen sind ein Hinweis, kein Test. Die CV-Werte sind Maxima über 21 Kandidaten (absolut leicht optimistisch). `C=1000` ist bei 13 von 24 Varianten der obere Rand des Rasters, bei `meta-only` ist `C=10` der untere und bei zwei Varianten `gamma=0.001`. Eine Referenz (RBF-SVM), nur CV auf dem Trainingsanteil.
